# Bronze Layer — Source Blob invoices/ → Bronze Volume (PDF Invoices)

**Day 6 | Manual / Scheduled — partitioned by date YYYY/MM/DD/**

Copies PDF invoice files from the `invoices/` folder in the instructor source blob into the Bronze Volume, preserving the `YYYY/MM/DD/` partition structure.

---

### Source layout

```
wasbs://source@dataenggdailystorage.blob.core.windows.net/
  └── invoices/
        └── YYYY/
              └── MM/
                    └── DD/
                          └── INV-AU-YYYY-NNNNNNNN.pdf
```

Current data: **June 2026** only, all 30 days, **~15 PDFs per day**, ~450 total.

Invoice number pattern: `INV-AU-2026-0002266` (Jun 1) → `INV-AU-2026-0002715` (Jun 30)

### Bronze Volume target

```
/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/
  └── invoices/
        └── YYYY/MM/DD/
              └── INV-AU-YYYY-NNNNNNNN.pdf
```

---

### Load modes — controlled by `LOAD_MODE` in Cell 2

| `LOAD_MODE` | What it copies |
|---|---|
| `full` | All invoices across all dates — use for first run |
| `daily` | Only the specific `YYYY/MM/DD` date folder |

## Cell 1 — Authenticate to source blob storage

In [ ]:
SCOPE = "kv-ev-scope"

STORAGE_ACCOUNT = dbutils.secrets.get(scope=SCOPE, key="source-storage-account")
CONTAINER       = dbutils.secrets.get(scope=SCOPE, key="source-container")
SAS_TOKEN       = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")

spark.conf.set(
    f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SAS_TOKEN
)

SOURCE_ROOT = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"

print(f"Storage account : {STORAGE_ACCOUNT}")
print(f"Container       : {CONTAINER}")
print(f"Source root     : {SOURCE_ROOT}")
print("Source blob authenticated — OK")

## Cell 2 — Set load mode and date

**This is the only cell you change between runs.**

- `LOAD_MODE = "full"` → copies all PDFs across all dates
- `LOAD_MODE = "daily"` → copies only `YYYY/MM/DD` date folder

In [ ]:
LOAD_MODE  = "daily"   # "full" or "daily"

LOAD_YEAR  = "2026"
LOAD_MONTH = "06"
LOAD_DAY   = "01"

BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
BASE_SUBPATH  = "invoices"

if LOAD_MODE == "full":
    SOURCE_PATH = f"{SOURCE_ROOT}/{BASE_SUBPATH}/"
    BRONZE_PATH = f"{BRONZE_VOLUME}/{BASE_SUBPATH}/"
    print(f"LOAD_MODE : full — all invoice dates")
elif LOAD_MODE == "daily":
    partition   = f"{LOAD_YEAR}/{LOAD_MONTH}/{LOAD_DAY}"
    SOURCE_PATH = f"{SOURCE_ROOT}/{BASE_SUBPATH}/{partition}/"
    BRONZE_PATH = f"{BRONZE_VOLUME}/{BASE_SUBPATH}/{partition}/"
    print(f"LOAD_MODE : daily — {partition}")
else:
    raise ValueError(f"Unknown LOAD_MODE: {LOAD_MODE}. Use 'full' or 'daily'.")

print(f"Source : {SOURCE_PATH}")
print(f"Bronze : {BRONZE_PATH}")

## Cell 3 — List source PDF files

In [ ]:
def list_files_recursive(path):
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return []
    files = []
    for item in items:
        if item.isDir():
            files.extend(list_files_recursive(item.path))
        else:
            files.append(item)
    return files

source_files = list_files_recursive(SOURCE_PATH)
pdf_files    = [f for f in source_files if f.name.endswith(".pdf")]

total_size_mb = sum(f.size for f in pdf_files) / (1024 * 1024)
print(f"PDF files found : {len(pdf_files)}")
print(f"Total size      : {round(total_size_mb, 1)} MB")
print()
for f in pdf_files[:20]:   # preview first 20
    rel = f.path.replace(SOURCE_PATH, "")
    print(f"  {rel:<60}  [{round(f.size/1024, 1)} KB]")
if len(pdf_files) > 20:
    print(f"  ... and {len(pdf_files) - 20} more")

## Cell 4 — Copy PDF files to Bronze Volume

Preserves the full `YYYY/MM/DD/` partition hierarchy. Folders created automatically by `dbutils.fs.cp`.

> PDFs are copied as-is. Content extraction (invoice number, amount, date) from PDF body is a
> Silver-layer concern. At Bronze we store the raw file — the file name itself carries the invoice ID.

In [ ]:
copied  = []
skipped = []

for file_info in pdf_files:
    relative_path = file_info.path.replace(SOURCE_PATH, "")
    dest_path     = BRONZE_PATH + relative_path
    try:
        dbutils.fs.cp(file_info.path, dest_path)
        copied.append(dest_path)
        if len(copied) <= 20 or len(copied) % 50 == 0:
            print(f"  COPIED  {relative_path}")
    except Exception as e:
        skipped.append((file_info.path, str(e)))
        print(f"  FAILED  {relative_path} — {e}")

print(f"\nResult: {len(copied)} copied, {len(skipped)} failed")
if skipped:
    raise Exception(f"{len(skipped)} file(s) failed — check output above.")

## Cell 5 — Verify files in Bronze Volume

In [ ]:
bronze_files = list_files_recursive(BRONZE_PATH)
bronze_pdfs  = [f for f in bronze_files if f.name.endswith(".pdf")]

status = "PASS" if len(bronze_pdfs) == len(pdf_files) else "FAIL"
print(f"[{status}] Source: {len(pdf_files)} PDFs  →  Bronze: {len(bronze_pdfs)} PDFs")

assert len(bronze_pdfs) == len(pdf_files), (
    f"Count mismatch — source: {len(pdf_files)}, bronze: {len(bronze_pdfs)}"
)
print("Verification passed — all PDFs confirmed in Bronze Volume.")

## Cell 6 — Invoice metadata from file names

The invoice number and date are encoded in the file path — no PDF parsing needed for basic metadata.

```
Path:  invoices/2026/06/01/INV-AU-2026-0002266.pdf
→ year=2026, month=06, day=01, invoice_id=INV-AU-2026-0002266
```

This cell builds a metadata DataFrame from file paths — useful for Silver layer audit tables.

In [ ]:
import re
from pyspark.sql import Row

rows = []
for f in bronze_pdfs:
    # Extract partition from path: .../invoices/YYYY/MM/DD/INV-AU-YYYY-NNNN.pdf
    m = re.search(r"invoices/(\d{4})/(\d{2})/(\d{2})/(.+\.pdf)$", f.path)
    if m:
        rows.append(Row(
            invoice_id  = m.group(4).replace(".pdf", ""),
            year        = m.group(1),
            month       = m.group(2),
            day         = m.group(3),
            file_size_kb= round(f.size / 1024, 1),
            bronze_path = f.path
        ))

df_meta = spark.createDataFrame(rows)
print(f"Invoice metadata rows: {df_meta.count():,}")
display(df_meta.orderBy("year", "month", "day").limit(20))

## Cell 7 — Summary

In [ ]:
print("=" * 60)
print("BRONZE INVOICE PDF MIGRATION — RUN SUMMARY")
print("=" * 60)
print(f"Load mode       : {LOAD_MODE}")
if LOAD_MODE == "daily":
    print(f"Date            : {LOAD_YEAR}/{LOAD_MONTH}/{LOAD_DAY}")
print(f"Source path     : {SOURCE_PATH}")
print(f"Bronze path     : {BRONZE_PATH}")
print(f"PDFs copied     : {len(copied)}")
print(f"PDFs failed     : {len(skipped)}")
print(f"Bronze total    : {len(bronze_pdfs)}")
print("=" * 60)
print("Next step: Silver layer parses PDF content or uses file-name metadata for invoice Delta table.")